In [ ]:
# Competition-Solution/notebooks/train/02_model_training/02_train_model_training_focal_loss.ipynb

### Submission Model Training

The notebooks within the **'02_model_training'** folder perform the training of a competition model based on the training dataset, focusing on the three subtasks:

- **Subtask 1:** Call2Action

- **Subtask 2:** Attacks on the Democratic Basic Order (FDGO)

- **Subtask 3:** Violence Detection


The goal of this model training is to implement and validate a class-weights + focal loss fine-tuning pipeline using a pre-trained Transformer model.

This approach will be used as the first submission to the GermEval2025 competition. The rationale behind using a transformer based model, specifically an encoder-based model, is that the task's challenges of coded language, implicit hate, and contextual nuances necessitate the bidirectional contextual understanding capabilities that encoder architectures provide, while modern encoder-based models offer improved efficiency through architectural innovations like sliding-window attention and extended sequence lengths that are particularly well-suited for processing the linguistic complexity found in extremist social media content. We are adding a focal loss to the class-weighted fine-tuning pipeline to focus on the hard examples, as our previous class weighted model seemed to struggle on the multiclass DBO subtask.

Note: This notebook heavily relies on our hierarchical .yaml config system and interactive jupyter notebook widgets. Instead of having a separate notebook for each subtask, we have a single notebook that can be configured to train the model for each subtask.
Please refer to the config files and the `Competition-Solution/configs/dir_info.md` for more information on this before using or reading through this code/notebook.

### Codabench Baseline Metrics

The Codabench contest provided baseline results to offer a starting benchmark. Their baselines were formed based on the full training dataset. The approaches and their reported Macro-F1 scores for each subtask are as follows:

*   **Subtask 1: Call2Action (Binary Classification)**
    *   **Baseline Approach:** Gradient-boosting classifier with SentenceBert embeddings and tweet polarity, using undersampling.
    *   **Reported Macro-F1 Score:** 0.59

*   **Subtask 2: Attacks on the Democratic Basic Order (FDGO) (Multi-class Classification)**
    *   **Baseline Approach:** Linear Support Vector Machine (SVM) with TF-IDF weighted bag-of-phrases (unigrams and bigrams), using a cost-sensitive SVM.
    *   **Reported Macro-F1 Score:** 0.47

*   **Subtask 3: Violence Detection (Binary Classification)**
    *   **Baseline Approach:** Large Language Model Qwen2.5 (32 billion parameters) in a few-shot scenario.
    *   **Reported Macro-F1 Score:** 0.69

These scores serve as an initial comparison point for the subsequent approaches and training runs.
These allow us to ballpark the performance of different approaches and motivate why we use an encoder-based transformer model for our main attempts.

---

##### <b>Imports</b>

In [19]:
import sys
from collections import Counter
from pathlib import Path
from rich.console import Console
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import wandb
import torch
import torch.nn as nn

# Hugging Face Imports
from transformers import (
  AutoProcessor,
  AutoModelForSequenceClassification,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
  EarlyStoppingCallback
)
from datasets import load_from_disk
import evaluate

console = Console()

# Path to this notebook
notebook_dir = Path.cwd()

# Project root directory
project_root_dir = notebook_dir.parent.parent.parent

# Source path
src_path = project_root_dir / "src"
sys.path.append(str(src_path))

console.print(f"Project root: {project_root_dir}", style="cyan")
console.print(f"Source path: {src_path}", style="cyan")
console.print(f"Source path exists: {src_path.exists()}", style="cyan")

# Local imports
import wandb_utils
import config_utils

Project root: /home/samuel/VSCode Projects/Bachelors-Thesis/Competition-Solution

Source path: /home/samuel/VSCode Projects/Bachelors-Thesis/Competition-Solution/src

Source path exists: True

### **Experiment Configuration**

In [20]:
# Default values for configuration choices
default_config_choices = {
    "CHOSEN_SUBTASK_DIR_NAME": "subtask1_call2action",
    "CHOSEN_EXPERIMENT_FILE_NAME": "ModernGBERT_train.yaml",
    "CHOSEN_DATASET_MODE": "train",
    "RUN_VERSION_TAG": "v3-best",
    "LOG_TO_WANDB": True,
    "USE_WANDB_SWEEPS": True,
}

# Initializes default config choices
global_config_choices = default_config_choices.copy()

console.print("Initial/Default Configuration Choices:", style="bold yellow")
for config_option, config_choice in global_config_choices.items():
    console.print(f"  {config_option}: {config_choice}")

Initial/Default Configuration Choices:

CHOSEN_SUBTASK_DIR_NAME: subtask1_call2action

CHOSEN_EXPERIMENT_FILE_NAME: ModernGBERT_train.yaml

CHOSEN_DATASET_MODE: train

RUN_VERSION_TAG: v3-best

LOG_TO_WANDB: True

USE_WANDB_SWEEPS: True

In [21]:
# Option values
subtask_options = [
    ("Sub-task 1 · Call-to-Action", "subtask1_call2action"),
    ("Sub-task 2 · Attacks on DBO", "subtask2_attacks_on_dbo"),
    ("Sub-task 3 · Violence", "subtask3_violence"),
]

experiment_options = [
    ("Euro-BERT · baseline", "EuroBERT_baseline.yaml"),
    ("Modern-GBERT · baseline", "ModernGBERT_baseline.yaml"),
    ("Modern-GBERT · train", "ModernGBERT_train.yaml"),
    ("Modern-GBERT · best", "ModernGBERT_best_from_sweep.yaml"),
]

dataset_mode_options = [
    ("Trial", "trial"),
    ("Train", "train"),
]

# Form Widgets
# Subtask Dropdown
dd_subtask = widgets.Dropdown(
    options=subtask_options,
    value=global_config_choices['CHOSEN_SUBTASK_DIR_NAME'],
    description="Sub-task:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Experiment Dropdown
dd_experiment = widgets.Dropdown(
    options=experiment_options,
    value=global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'],
    description="Experiment:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Dataset Mode Dropdown
dd_mode = widgets.Dropdown(
    options=dataset_mode_options,
    value=global_config_choices['CHOSEN_DATASET_MODE'],
    description="Dataset mode:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Run Version Text Input
txt_version = widgets.Text(
    value=global_config_choices['RUN_VERSION_TAG'],
    description="Run tag:",
    placeholder="e.g. v1",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="200px")
)

# Log to W&B Checkbox
log_to_wandb = widgets.Checkbox(
    description="Log to W&B",
    value=global_config_choices['LOG_TO_WANDB'],
    layout=widgets.Layout(width="200px"),
    style={"description_width": "90px"}
)

# W&B Sweep Checkbox
log_to_wandb_sweep = widgets.Checkbox(
    description="W&B Sweep",
    value=global_config_choices['USE_WANDB_SWEEPS'],
    layout=widgets.Layout(width="200px"),
    style={"description_width": "90px"}
)

# Save Button
save_btn = widgets.Button(
    description="Save choices",
    button_style="success",
    icon="check"
)

status_out = widgets.Output()

def _on_run_clicked(b):
    global global_config_choices
    global_config_choices = {
        "CHOSEN_SUBTASK_DIR_NAME": dd_subtask.value,
        "CHOSEN_EXPERIMENT_FILE_NAME": dd_experiment.value,
        "CHOSEN_DATASET_MODE": dd_mode.value,
        "RUN_VERSION_TAG": txt_version.value,
        "LOG_TO_WANDB": log_to_wandb.value,
        "USE_WANDB_SWEEPS": log_to_wandb_sweep.value
    }
    with status_out:
        clear_output(wait=True)
        console.print("Config saved:", global_config_choices, style="green")

save_btn.on_click(_on_run_clicked)

# Builds the form
form = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Configure experiment</h4>"),
    dd_subtask,
    dd_experiment,
    dd_mode,
    txt_version,
    log_to_wandb,
    log_to_wandb_sweep,
    save_btn,
    status_out
])

display(form)

In [22]:
# Path to the configs directory
config_base_dir_nb = project_root_dir / "configs"

# Defines the paths to the global, subtask and experiment configs
base_cfg_path = config_base_dir_nb / "base.yaml" # Global base config
subtask_base_cfg_path = config_base_dir_nb / global_config_choices['CHOSEN_SUBTASK_DIR_NAME'] / "base.yaml" # Subtask base config
experiment_cfg_path = config_base_dir_nb / global_config_choices['CHOSEN_SUBTASK_DIR_NAME'] / global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'] # Experiment config

# Loads the global, subtask and experiment configs
cfg = config_utils.load_config(
  base_config_path=base_cfg_path,
  subtask_config_path=subtask_base_cfg_path,
  experiment_config_path=experiment_cfg_path
)

# Raises an error if one of the config files is not found
if not cfg:
  raise ValueError(f'Configuration files could not be loaded. Please check the config YAML files in the \"configs\" directory and widget selections:\n'
                   f"    Global base config: {base_cfg_path}\n"
                   f"    Subtask base config: {subtask_base_cfg_path}\n"
                   f"    Experiment config: {experiment_cfg_path}")

console.print(f"Successfully loaded and merged configurations for: \n"
              f"{global_config_choices['CHOSEN_SUBTASK_DIR_NAME']} / "
              f"{global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME']} "
              f"(Mode: {global_config_choices['CHOSEN_DATASET_MODE']})", style="green")

Successfully loaded and merged configurations for: 
subtask1_call2action / ModernGBERT_train.yaml (Mode: train)

##### <b>Loading the HF datasets</b>

In [23]:
# Defines the path to the processed data directory
processed_data_root_dir = project_root_dir / cfg['paths']['processed_data_dir_name']
console.print(f"Processed Data Root Directory: '{processed_data_root_dir}'", style="bold")

# Defines the suffix of the dataset (e.g. dbo/dbo_hf_dataset)
dataset_load_path_suffix = cfg['dataset_details']['hf_dataset_path_suffix']
console.print(f"Dataset Load Path Suffix: '{dataset_load_path_suffix}'", style="bold")

# Constructs the full path to the dataset by joining the processed data root directory and the dataset load path suffix
full_dataset_load_path = processed_data_root_dir / dataset_load_path_suffix

console.print(f"Loading dataset for subtask '{cfg['subtask_name']}' from: '{full_dataset_load_path}'", style="green")

# Tries to load the dataset from the full path
try:
    raw_dataset = load_from_disk(str(full_dataset_load_path))
    console.print(f"Successfully loaded dataset: \n{raw_dataset}", style="green")
except FileNotFoundError:
    console.print(f"ERROR: Dataset not found at {full_dataset_load_path}. Please check your config and data.", style="bold red")
    raise
except Exception as e:
    console.print(f"ERROR: Could not load dataset from {full_dataset_load_path}: {e}", style="bold red")
    raise

Processed Data Root Directory: '/home/samuel/VSCode Projects/Bachelors-Thesis/Competition-Solution/data/processed'

Dataset Load Path Suffix: 'c2a/c2a_hf_dataset_train'

Loading dataset for subtask 'call2action' from: '/home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/data/processed/c2a/c2a_hf_dataset_train'

Successfully loaded dataset: 
DatasetDict({
    train: Dataset({
        features: ['id', 'description', 'C2A'],
        num_rows: 5472
    })
    validation: Dataset({
        features: ['id', 'description', 'C2A'],
        num_rows: 1331
    })
    test: Dataset({
        features: ['id', 'description', 'C2A'],
        num_rows: 2982
    })
})

In [24]:
# Model configuration and paths
console.print(f"Model: {cfg['model_checkpoint']}", style="bold")
console.print(f"Model config: {cfg['model_config']}", style="bold")

# Checks for local model first, fallback to HF Hub
local_model_path = project_root_dir / cfg['paths']['base_models_dir_name'] / cfg['model_checkpoint']
if local_model_path.exists():
    model_path = str(local_model_path)
    console.print(f"Found local model at: {model_path}", style="cyan")
else:
    model_path = cfg['model_checkpoint']
    console.print(f"Local model not found, using Hugging Face Hub: {model_path}", style="yellow")

# Loads the model
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        **cfg['model_config']
    )
    console.print(f"Successfully loaded model: {model.__class__.__name__}", style="green")
except Exception as e:
    console.print(f"ERROR: Could not load model from {model_path}: {e}", style="bold red")
    raise

# If a gpu is available, moves the model to the gpu
if torch.cuda.is_available():
    model.to("cuda")
    console.print("Model moved to GPU", style="green")

Model: LSX-UniWue/ModernGBERT_134M

Model config: {'num_labels': 2, 'id2label': {0: 'False', 1: 'True'}, 'label2id': {'False': 0, 'True': 1}, 
'trust_remote_code': True}

Found local model at: /home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/models/base_models/LSX-UniWue/ModernGBERT_134M

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at /home/samuel/VSCode Projects/Bachelors-Thesis/Competition-Solution/models/base_models/LSX-UniWue/ModernGBERT_134M and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Successfully loaded model: ModernBertForSequenceClassification

RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
console.print(f"Loading AutoProcessor for '{model_path}'", style="green")

# Loads the processor and adding the special tokens (Anonymization tokens that we set during the data preprocessing phase)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
processor.add_special_tokens({"additional_special_tokens": cfg['tokenization']['special_tokens']})
model.resize_token_embeddings(len(processor), mean_resizing=True)
console.print(f"Processor loaded with {len(cfg['tokenization']['special_tokens'])} special tokens, total: {len(processor)}", style="cyan")

Loading AutoProcessor for '/home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/models/base_models/LSX-UniWue/ModernGBERT_134M'

Processor loaded with 6 special tokens, total: 31108

Total number of unique tokens: **128262**

Dimensionality of the token embeddings: **768**

ID of padding token: **128001**

In [ ]:
console.print(f"Tokenizing dataset for subtask '{cfg['subtask_name']}'...", style="green")

# Tokenization function
def tokenize_function(examples):
    return processor(
        examples[cfg['tokenization']['text_column_name']],
        padding=cfg['tokenization']['padding'],
        truncation=cfg['tokenization']['truncation'],
    )

# Applies the tokenization mapping function to the raw dataset in batches
try:
    tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
    console.print(f"Dataset tokenized successfully: {tokenized_dataset}", style="bold green")
except Exception as e:
    console.print(f"ERROR during dataset tokenization: {e}", style="bold red")
    raise

# Renames the subtasks respective label column to "labels", as the Trainer expects it
tokenized_dataset = tokenized_dataset.rename_column(cfg['dataset_details']['label_column'], "labels")
console.print(f"Renamed '{cfg['dataset_details']['label_column']}' to \"labels\"", style="green")

# Saves the tokenized dataset to disk
save_path = processed_data_root_dir / cfg['dataset_details']['hf_dataset_tokenized_path_suffix']
tokenized_dataset.save_to_disk(str(save_path))
console.print(f"Tokenized dataset saved to: {save_path}", style="green")

Tokenizing dataset for subtask 'call2action'...

Map:   0%|          | 0/5472 [00:00<?, ? examples/s]

ERROR during dataset tokenization: name 'processor' is not defined

NameError: name 'processor' is not defined

In [ ]:
# Defines the data collator that is used to collate the tokenized dataset (per batch)
# (Pads the tokenized inputs to the same length, Stacks the tokenized inputs into a batch, Returns the tokenized inputs as PyTorch tensors)
data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    return_tensors="pt"
)
console.print("Data collator configured for PyTorch tensors", style="cyan")

Data collator configured for PyTorch tensors

- [Full List of Training Arguments](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)

In [ ]:
# Builds the run name and directories
dataset_mode = global_config_choices['CHOSEN_DATASET_MODE'] # e.g. 'trial' or 'train'
run_version = global_config_choices['RUN_VERSION_TAG'] # e.g. 'v1'

# Builds the run name e.g. 'train-c2a-moderngbert-train-v1'
run_name = f"{cfg['dataset_modes'][dataset_mode]['suffix']}-{cfg['subtask_id']}-{cfg['wandb_run_name_parts']['model_arch']}-{cfg['experiment_type']}"
if run_version:
    run_name += f"-{run_version}"

# Builds the output and logging directory paths
output_dir = project_root_dir / cfg['paths']['output_dir_base'] / cfg['subtask_id'] / run_name
logging_dir = project_root_dir / cfg['paths']['logging_dir_base'] / dataset_mode / run_name

console.print(f"Training run: {run_name}", style="bold yellow")
console.print(f"Early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="cyan")

# Creates the training arguments (Uses the configs settings)
training_args = TrainingArguments(
    output_dir=str(output_dir),
    logging_dir=str(logging_dir),
    run_name=run_name,
    **cfg['training_arguments']
)

console.print(f"Training arguments: {training_args}", style="cyan")

Training run: train-vio-moderngbert-best-v3-best-fl

Early stopping patience: 5

Training arguments: TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler':
True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
eval_use_gather_object=False,
fp16=True,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_v2': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=True,
gradient_checkpointing_kwargs={'use_reentrant': False},
greater_is_better=True,
group_by_length=False,
half_precision_backend=auto,
hub_always_push=False,
hub_model_id=None,
hub_private_repo=None,
hub_strategy=every_save,
hub_token=<HUB_TOKEN>,
ignore_data_skip=False,
include_for_metrics=[],
include_inputs_for_metrics=False,
include_num_input_tokens_seen=False,
include_tokens_per_second=False,
jit_mode_eval=False,
label_names=None,
label_smoothing_factor=0.0,
learning_rate=3e-05,
length_column_name=length,
load_best_model_at_end=True,
local_rank=0,
log_level=passive,
log_level_replica=warning,
log_on_each_node=True,
logging_dir=/home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/logs/train/train-vio-moderngbert-best-v3-best-fl,
logging_first_step=True,
logging_nan_inf_filter=True,
logging_steps=100,
logging_strategy=steps,
lr_scheduler_kwargs={},
lr_scheduler_type=linear,
max_grad_norm=None,
max_steps=-1,
metric_for_best_model=eval_f1-macro,
mp_parameters=,
neftune_noise_alpha=None,
no_cuda=False,
num_train_epochs=3,
optim=adamw_torch,
optim_args=None,
optim_target_modules=None,
output_dir=/home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/models/finetuned_models/vio/train-vio-moderngbert-best-v3-best-fl,
overwrite_output_dir=False,
past_index=-1,
per_device_eval_batch_size=32,
per_device_train_batch_size=16,
prediction_loss_only=False,
push_to_hub=False,
push_to_hub_model_id=None,
push_to_hub_organization=None,
push_to_hub_token=<PUSH_TO_HUB_TOKEN>,
ray_scope=last,
remove_unused_columns=True,
report_to=['wandb', 'tensorboard'],
restore_callback_states_from_checkpoint=False,
resume_from_checkpoint=None,
run_name=train-vio-moderngbert-best-v3-best-fl,
save_on_each_node=False,
save_only_model=False,
save_safetensors=True,
save_steps=500,
save_strategy=epoch,
save_total_limit=0,
seed=42,
skip_memory_metrics=True,
tf32=None,
torch_compile=False,
torch_compile_backend=None,
torch_compile_mode=None,
torch_empty_cache_steps=None,
torchdynamo=None,
tpu_metrics_debug=False,
tpu_num_cores=None,
use_cpu=False,
use_ipex=False,
use_legacy_prediction_loop=False,
use_liger_kernel=False,
use_mps_device=False,
warmup_ratio=0.0,
warmup_steps=100,
weight_decay=0.08113106416454527,
)

In [ ]:
# Loads the evaluation metrics from the config
loaded_metrics = {}

for metric in cfg['evaluation']['metrics_to_load']:
    loaded_metrics[metric] = evaluate.load(metric)
for metric in loaded_metrics:
    console.print(f"Loaded metric: {metric}", style="green")

# Defines the compute_metrics function that is used to compute the evaluation metrics during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    return {
        "f1-macro": loaded_metrics['f1'].compute(predictions=preds, references=labels, average=cfg['evaluation']['f1_average_type'])['f1'],
        "accuracy": loaded_metrics['accuracy'].compute(predictions=preds, references=labels)['accuracy'],
        "precision-macro": loaded_metrics['precision'].compute(predictions=preds, references=labels, average=cfg['evaluation']['precision_average_type'])['precision'],
        "recall-macro": loaded_metrics['recall'].compute(predictions=preds, references=labels, average=cfg['evaluation']['recall_average_type'])['recall']
    }

Loaded metric: f1

Loaded metric: accuracy

Loaded metric: precision

Loaded metric: recall

In [ ]:
# Custom Trainer that implements class weights + focal loss for class imbalance mitigation

# Focal loss will guide the model to focus on the hard examples

# Regular cross entropy loss formula: -log(probability_of_correct_class)
# Focal loss formula: class_weight * difficulty_multiplier * cross_entropy_loss
# difficulty_multiplier = (1 - confidence)^gamma
# Gamma is usually just set to 2.0 as this is what is suggested in the original paper (https://arxiv.org/pdf/1708.02002.pdf)
# Higher gamma values focus more on the hard examples

# Original Formula from paper: FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
# Where: 
# p_t = confidence_for_correct_class
# α_t = (alpha) class_weight
# γ = gamma

# Example: Model predicts 0.9 for the correct class, 0.1 for the incorrect class
# confidence_for_correct_class = 0.9
# difficulty_multiplier = (1 - 0.9)^2 = 0.1^2 = 0.01
# cross_entropy_loss = -log(0.9) = 0.105
# focal_loss = class_weight * 0.01 * 0.105 = very small loss
# -> This is an EASY example, so focal loss almost ignores it

# Counter-example: Model predicts 0.3 for the correct class, 0.7 for the incorrect class  
# confidence_for_correct_class = 0.3
# difficulty_multiplier = (1 - 0.3)^2 = 0.7^2 = 0.49
# cross_entropy_loss = -log(0.3) = 1.204
# focal_loss = class_weight * 0.49 * 1.204 = much larger loss
# -> This is a HARD example, so focal loss heavily focuses on it

# As a result, the model spends more time learning from mistakes on hard examples
# rather than wasting time on examples it already gets right

# This is in addition to the class weights, which are used to balance the classes

# Inherits from the Trainer class to implement focal loss
# This will ensure that the model focuses more on the hard examples
# and less on the easy examples

# HF Trainer Object Docs: https://huggingface.co/docs/transformers/main/en/main_classes/trainer
class FocalLossTrainer(Trainer):
    
    # We pass the class weights and gamma as instance variables
    def __init__(self, class_weights=None, gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    # Overrides the compute_loss method of the standard Trainer class to implement focal loss
    # This is called once per batch during training
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels") # Extracts the true labels from the batch
        outputs = model(**inputs) # Forwards pass through the model
        logits = outputs.get("logits") # Gets the raw predictions from the models last layer (unnormalized & before softmax activation)

        # Converts the logits to probabilities using softmax, essentially normalizing them to sum up to 1
        probabilities = torch.softmax(logits, dim=-1)
        
        # Gets the probability for the correct class for each sample
        # This tells us how confident the model is about the correct answer
        confidence_for_correct_class = probabilities.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
        
        # Calculates the difficulty multiplier: (1 - confidence)^gamma
        # High confidence (0.9) -> (1-0.9)^2 = 0.01 (easy, ignore)
        # Low confidence (0.3) -> (1-0.3)^2 = 0.49 (hard, focus)
        difficulty_multiplier = (1 - confidence_for_correct_class) ** self.gamma

        # If class weights are provided, we use them to calculate the focal loss
        if self.class_weights is not None:
            # Converts the class weights from Python list to PyTorch tensor
            # A tensor is essentially a multi-dimensional array
            # Ensures tensor is on same device (CPU/GPU) as model and uses the appropriate data type 
            # (float16 as we are using half precision for training, so the class weights need to be in half precision too, but logits.dtype infers it from the models logits anyway)
            weight_tensor = torch.tensor(self.class_weights, dtype=logits.dtype, device=logits.device)

            # Creates the loss function with our class weights (cross entropy loss)
            # Internally, the following happens:
            # For each sample in the batch:
            # true_class = labels[i]
            # raw_loss = -ln(softmax(logits[i])[true_class])
            # weighted_loss = raw_loss * class_weights[true_class]

            # We use reduction="none" instead of the default "mean" to get the loss for each sample in the batch individually
            # We want to discern individual hard tweets from easy tweets and treat them differently
            # Compared to class weights, where each tweet of the same class is treated equally
            loss_fct = nn.CrossEntropyLoss(weight=weight_tensor, reduction='none')
        else:
            # If no class weights are provided, we use the default loss function
            # We use reduction="none" so the focal modulator can scale each sample individually 
            loss_fct = nn.CrossEntropyLoss(reduction='none')

        # Calculates the cross entropy loss for each sample in the batch using the loss function defined above
        # Reshapes tensors to the format CrossEntropyLoss expects:
        # logits.view(-1, num_labels) -> 2D: [batch_size, num_classes]  
        # labels.view(-1) -> 1D: [batch_size]
        # The -1 tells PyTorch to infer that dimension automatically
        # https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
        cross_entropy_loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        # Applies the focal loss: multiplies cross entropy by difficulty multiplier
        # Again: difficulty_multiplier is (1 - confidence)^gamma
        # cross_entropy_loss is -log(probability_of_correct_class)
        # So, focal_loss = (1 - confidence)^gamma * -log(probability_of_correct_class)
        focal_loss = difficulty_multiplier * cross_entropy_loss
        
        # Takes the mean over the batch to get the final loss value
        # Gradient descent expects a single scalar loss value per batch to calculate gradients
        # The mean represents the average "wrongness" of the model's predictions for this batch
        # which is then used to update all model parameters in the direction that reduces this error
        final_loss = focal_loss.mean()
        
        return (final_loss, outputs) if return_outputs else final_loss

# Calculates the class weights
def calculate_class_weights(dataset, label_column="label"):
    # Extracts the labels from the dataset
    labels = dataset[label_column]

    # Counter comes from the python standard collections library and counts the occurences of each element in the list
    # e.g. if labels is [0, 0, 1, 1, 2, 2, 2], class_counts will be {0: 2, 1: 2, 2: 3}
    class_counts = Counter(labels)
    num_classes = len(class_counts)
    total_samples = len(labels)

    # Calculates the weights: total_samples / (num_classes * samples_in_class_i)
    class_weights = []

    # Loops through the classes and calculates the weights using the formula above
    for class_id in sorted(class_counts.keys()):
        weight = total_samples / (num_classes * class_counts[class_id])
        class_weights.append(weight)

    return class_weights

# Prints the class weights and their distribution
def print_class_weights_info(class_weights, dataset, label_column="label"):
    labels = dataset[label_column]
    class_counts = Counter(labels)
    total_samples = len(labels)

    console.log("Class Distribution and Weights:")
    
    for class_id in sorted(class_counts.keys()):
        weight = class_weights[class_id]
        console.log(f"Class {class_id}: {class_counts[class_id]} samples, Weight: {weight:.4f}")

    console.log(f"\nTotal Samples: {total_samples}")
    console.log(f"Number of Classes: {len(class_counts)}")

In [ ]:
# Clears the GPU cache (Useful when doing multiple subsequent runs on the same machine and with limited computational resources)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Calculates the class weights (see method that we defined in above cell)
class_weights = calculate_class_weights(tokenized_dataset[cfg['dataset_splits']['train']], label_column="labels")
print_class_weights_info(class_weights, tokenized_dataset[cfg['dataset_splits']['train']], label_column="labels")

# Gets gamma from config (with fallback)
gamma = cfg.get('gamma', 2.0)
console.print(f"Using gamma value: {gamma}", style="cyan")

# Initializes the FocalLossTrainer that we defined above (Takes the same arguments as the standard Trainer class + class weights and gamma)
trainer = FocalLossTrainer(
    model=model,
    processing_class=processor,
    data_collator=data_collator,
    args=training_args,
    train_dataset=tokenized_dataset[cfg['dataset_splits']['train']],
    eval_dataset=tokenized_dataset[cfg['dataset_splits']['validation']],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['callbacks']['early_stopping_patience'])],
    class_weights=class_weights,
    gamma=gamma
)

console.print(f"Trainer initialized with early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="green")
console.print(f"Training on {len(tokenized_dataset[cfg['dataset_splits']['train']])} samples", style="cyan")
console.print(f"Validating on {len(tokenized_dataset[cfg['dataset_splits']['validation']])} samples", style="cyan")

[17:28:44] Class Distribution and Weights:                                                        ]8;id=645150;file:///tmp/ipykernel_618408/1883306959.py\1883306959.py]8;;\:]8;id=247782;file:///tmp/ipykernel_618408/1883306959.py#131\131]8;;\

           Class 0: 5775 samples, Weight: 0.5390                                                  ]8;id=269325;file:///tmp/ipykernel_618408/1883306959.py\1883306959.py]8;;\:]8;id=739680;file:///tmp/ipykernel_618408/1883306959.py#134\134]8;;\

           Class 1: 451 samples, Weight: 6.9024                                                   ]8;id=829097;file:///tmp/ipykernel_618408/1883306959.py\1883306959.py]8;;\:]8;id=186889;file:///tmp/ipykernel_618408/1883306959.py#134\134]8;;\

                                                                                                  ]8;id=397011;file:///tmp/ipykernel_618408/1883306959.py\1883306959.py]8;;\:]8;id=576851;file:///tmp/ipykernel_618408/1883306959.py#136\136]8;;\
           Total Samples: 6226                                                                                     

           Number of Classes: 2                                                                   ]8;id=924439;file:///tmp/ipykernel_618408/1883306959.py\1883306959.py]8;;\:]8;id=10411;file:///tmp/ipykernel_618408/1883306959.py#137\137]8;;\

Using gamma value: 0.5191325114827803

Trainer initialized with early stopping patience: 5

Training on 6226 samples

Validating on 1557 samples

In [26]:
# Starts the Tensorboard Dashboard in the Jupyter Notebook (Requires the Tensorboard IDE extension)
# Otherwise, the Tensorboard Dashboard is also available at http://localhost:6006/
%load_ext tensorboard
%tensorboard --logdir "{str(logging_dir)}"

Reusing TensorBoard on port 6006 (pid 64403), started 14:33:09 ago. (Use '!kill 64403' to kill it.)

In [ ]:
# Finishes any existing W&B run
if wandb.run is not None:
    console.print("Finishing previous W&B run...", style="yellow")
    wandb.finish()

# Initializes W&B if we chose to log to W&B in the jupyter widget at the beginning of the notebook
if global_config_choices['LOG_TO_WANDB']:
    if global_config_choices['USE_WANDB_SWEEPS']:
        console.print("W&B Sweeps enabled. Sweep will handle run initialization.", style="yellow")
        wandb_run = None
    else:
        console.print("W&B logging enabled. Attempting login...", style="yellow")
        
        # Logs in to W&B and initializes run if successful
        if wandb_utils.login_wandb():
            console.print("W&B login successful.", style="bold green")
            
            # Initializes the W&B run
            try:
                wandb_run = wandb_utils.init_wandb(
                    loaded_config=cfg,
                    dataset_mode=global_config_choices['CHOSEN_DATASET_MODE'],
                    logging_dir=project_root_dir / cfg['paths']['logging_dir_base'],
                    version_tag=global_config_choices['RUN_VERSION_TAG']
                )
                
                # Adds W&B to training args and sets up model watching
                if "wandb" not in training_args.report_to:
                    training_args.report_to.append("wandb")
                    console.print("W&B added to training args", style="green")
                
                # Sets up model watching
                wandb_run.watch(
                    models=model,
                    log=cfg['wandb']['watch_model_log'], 
                    log_freq=cfg['wandb']['watch_model_log_freq']
                )
                console.print(f"W&B model watching enabled: {cfg['wandb']['watch_model_log']} (freq: {cfg['wandb']['watch_model_log_freq']})", style="cyan")
                console.print(f"W&B run initialized: {wandb_run.name} (ID: {wandb_run.id})", style="bold green")
                
            except Exception as e:
                console.print(f"ERROR: Failed to initialize W&B run: {e}", style="bold red")
                wandb_run = None
        else:
            console.print("W&B login failed. Training will proceed without W&B logging.", style="bold red")
            wandb_run = None
else:
    console.print("W&B logging disabled in the widget.", style="yellow")
    wandb_run = None

W&B logging enabled. Attempting login...

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/samuel/.netrc
wandb: Currently logged in as: samuel-ruairi-bullard (uni-regensburg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to W&B


W&B login successful.

Initialized W&B run: train-vio-moderngbert-best-v3-best-fl (Group: train/violence/moderngbert, ID: atwbs5si)


W&B model watching enabled: all (freq: 1000)

W&B run initialized: train-vio-moderngbert-best-v3-best-fl (ID: atwbs5si)

In [ ]:
# Helper method for using the Weights & Biases Sweeps feature if the user has selected to use them in the jupyter widget at the beginning of the notebook
def train_with_sweeps():  
    # W&B automatically initializes the run and populates wandb.config
    wandb.init()
    
    # Updates the config with the sweep parameters
    updated_cfg = wandb_utils.update_config_with_sweep_params(cfg)

    # Gets gamma from config (with fallback)
    gamma = wandb.config.get('gamma', cfg.get('gamma', 2.0))
    
    # Logs the gamma value for tracking to Weights & Biases
    wandb.log({"gamma": gamma})
    console.print(f"Sweep using gamma value: {gamma}", style="cyan")

    # Gets the current dataset mode
    current_dataset_mode = global_config_choices['CHOSEN_DATASET_MODE']

    # Checks if the local model path exists, if not, uses the model checkpoint
    if local_model_path.exists():
        sweep_model_path = str(local_model_path)
    else:
        sweep_model_path = cfg['model_checkpoint']
    
    # Loads the model
    sweep_model = AutoModelForSequenceClassification.from_pretrained(
        sweep_model_path,
        **cfg['model_config']
    )
    
    # Moves the model to GPU if available
    if torch.cuda.is_available():
        sweep_model.to("cuda")
    
    # Resizes the token embeddings to match the processor
    sweep_model.resize_token_embeddings(len(processor), mean_resizing=True)

    # Creates unique directories for each sweep run
    sweep_run_name = f"{run_name}-sweep-{wandb.run.name}"
    sweep_output_dir = project_root_dir / cfg['paths']['output_dir_base'] / cfg['subtask_id'] / sweep_run_name
    sweep_logging_dir = project_root_dir / cfg['paths']['logging_dir_base'] / current_dataset_mode / sweep_run_name

    # Removes the warning when training (Sweeps already logs to W&B)
    training_args_config = updated_cfg['training_arguments'].copy()

    # Keeps tensorboard but removes wandb to avoid conflicts
    if 'report_to' in training_args_config:
        training_args_config['report_to'] = [
            reporter for reporter in training_args_config['report_to'] 
            if reporter != "wandb"
        ]
    
    # Creates the training arguments for the sweep run
    sweep_training_args = TrainingArguments(
        output_dir=str(sweep_output_dir),
        logging_dir=str(sweep_logging_dir),
        run_name=sweep_run_name,
        **updated_cfg['training_arguments']
    )
    
    # Uses the sweep_model instead of the original model
    sweep_trainer = FocalLossTrainer(
        model=sweep_model,
        processing_class=processor,
        data_collator=data_collator,
        args=sweep_training_args,
        train_dataset=tokenized_dataset[cfg['dataset_splits']['train']],
        eval_dataset=tokenized_dataset[cfg['dataset_splits']['validation']],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['callbacks']['early_stopping_patience'])],
        class_weights=class_weights,
        gamma=gamma  # Use gamma from sweep
    )
    
    # Trains the model and returns the best metric for optimization
    result = sweep_trainer.train()
    
    # Logs the best metric for sweep optimization
    best_metric = result.metrics.get(f"eval_{cfg['training_arguments']['metric_for_best_model']}", 0)
    wandb.log({"best_metric": best_metric})
    
    return best_metric


In [18]:
# Regular training function
def train_single_run():
    # Tries to train the model
    try:
        training_result = trainer.train()
        console.print("Training finished successfully!", style="bold green")
        if training_result.metrics:
            console.print("Final metrics:", style="cyan")
            for metric, metric_value in training_result.metrics.items():
                console.print(f"  {metric}: {metric_value:.4f}", style="white")
    except Exception as e:
        console.print(f"Training failed with error: {e}", style="bold red")
        raise


In [ ]:
# Creates a jupyter widget button to start the training loop
train_btn = widgets.Button(
    description="Train model",
    icon="play",
    button_style="success",
    tooltip="Start the training loop",
    layout=widgets.Layout(width="160px")
)

# Creates a jupyter widget output to display the training logs
log_out = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc",
                    max_height="350px",
                    overflow="auto",
                    padding="4px")
)
# Defines the function that is called when the training button is clicked
def _train_model(btn):
    # Disables the training button after starting the training run
    train_btn.disabled = True
    
    # Clears the log output when starting a new training run
    with log_out:
        clear_output(wait=True)
        
        # If the user has selected to use W&B Sweeps, creates and runs a sweep
        if global_config_choices['USE_WANDB_SWEEPS']:
            console.print("Starting W&B Sweep...", style="bold green")
            
            # Checks if a sweep config exists in the respective config .yaml file
            if "wandb_sweep_config" not in cfg:
                console.print("ERROR: No sweep configuration found in config file", style="bold red")

                # Enables the training button again if no sweep config exists
                train_btn.disabled = False
                return
            
            try:
                # Logs in to W&B
                if not wandb_utils.login_wandb():
                    console.print("ERROR: W&B login failed", style="bold red")
                    train_btn.disabled = False
                    return
                
                # Creates and runs a sweep
                console.print("Creating sweep...", style="yellow")
                sweep_id = wandb_utils.create_and_run_sweep(cfg, count=10, train_with_sweeps=train_with_sweeps)
                console.print(f"Sweep completed! Sweep ID: {sweep_id}", style="bold green")
                
            except Exception as e:
                console.print(f"Sweep failed with error: {e}", style="bold red")
                raise
        
        # If the user did not select to use W&B Sweeps, trains the model normally
        else:
            console.print("Training started...", style="bold green")
            train_single_run()
    
    # Enables the training button again
    train_btn.disabled = False

# Adds the training button to the form
train_btn.on_click(_train_model)

# Displays the training button and log output
display(widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Run experiment</h4>"),
    train_btn,
    log_out
]))


In [ ]:
# Saves the best model (This must not be run before the "Train model" widget has finished, as it will interrupt the trianing!)
trainer.save_model(f"{str(output_dir)}/best_model_{cfg['training_arguments']['metric_for_best_model']}")

In [ ]:
# Saves and uploads a W&B artifact of the finetuned model (Uploads the finetuned model to W&B)
if wandb_run is not None:
    try:
        console.print("Saving model and creating W&B artifact...", style="yellow")
        
        # Builds the artifact name, description, and target path
        artifact_name = cfg['wandb']['artifact']['name_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id'],
            experiment_type=cfg['experiment_type']
        ) # e.g. "moderngbert-c2a-train"
        
        description = cfg['wandb']['artifact']['description_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_name=cfg['subtask_name'],
            experiment_type=cfg['experiment_type'],
            dataset_mode_suffix=cfg['dataset_modes'][global_config_choices['CHOSEN_DATASET_MODE']]['suffix']
        ) # e.g. "Fine-tuned moderngbert for call2action (GermEval 2025)"
        
        # Ensures the target path is in "collection/alias" or "project/collection/alias" format for linking
        target_path = cfg['wandb']['artifact']['portfolio_path_template'].format(
            project=cfg['wandb']['project'],
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id']
        ) # e.g. Bachelors-Thesis/moderngbert_c2a_models"
        
        # Prints the artifact name, description, local model path, and target W&B path for linking
        console.print(f"Creating artifact: {artifact_name}", style="cyan")
        console.print(f"Description: {description}", style="cyan")
        console.print(f"Local model path: {output_dir}", style="cyan")
        console.print(f"Target W&B path for linking: {target_path}", style="cyan")
        
        # Saves the model to W&B
        artifact = wandb_utils.save_and_upload_model_to_wandb(
            run=wandb_run,
            name=artifact_name,
            model_type=cfg['wandb']['artifact']['model_type'],
            description=description,
            metadata={
                'model_checkpoint': cfg['model_checkpoint'],
                'subtask_id': cfg['subtask_id'],
                'subtask_name': cfg['subtask_name'],
                'experiment_type': cfg['experiment_type'],
                'dataset_mode': global_config_choices['CHOSEN_DATASET_MODE'],
                'run_name': wandb_run.name,
                'training_args': cfg['training_arguments'],
                'final_metrics': trainer.state.log_history[-1] if trainer.state.log_history else {}
            },
            local_path=str(output_dir), # Path to the local directory containing the model files
            target_path=target_path # Path in W&B to link this artifact version
        )
        
    except Exception as e:
        console.print(f"ERROR: Failed to save model artifact to W&B: {e}", style="bold red")
        console.print("Model was saved locally but W&B artifact creation/linking failed.", style="yellow")
    
    # Finishes the W&B run after the model has been saved
    finally:
        console.print("Finishing W&B run...", style="yellow")
        wandb_run.finish()
        console.print("W&B run finished.", style="green")
else:
    console.print("No W&B run to finish (W&B was disabled or failed to initialize).", style="yellow")

console.print(f"Training completed! Model saved to: {target_path}", style="bold green")

Saving model and creating W&B artifact...

Creating artifact: moderngbert-vio-best

Description: Fine-tuned moderngbert for violence (GermEval 2025)

Local model path: /home/samuel/VSCode 
Projects/Bachelors-Thesis/Competition-Solution/models/finetuned_models/vio/train-vio-moderngbert-best-v3-best-fl

Target W&B path for linking: Bachelors-Thesis/moderngbert_vio_models

...

Successfully created and uploaded W&B artifact: moderngbert-vio-best:v0


Finishing W&B run...

eval/accuracy,█▁▄
eval/f1-macro,█▁▅
eval/loss,▁▃█
eval/precision-macro,█▁▄
eval/recall-macro,▇▁█
eval/runtime,▁▄█
eval/samples_per_second,█▅▁
eval/steps_per_second,█▅▁
train/epoch,▁▂▂▃▃▃▄▅▅▆▆▆▇███
train/global_step,▁▂▂▃▃▃▄▅▅▆▆▆▇███
train/learning_rate,▁██▇▆▆▅▄▄▃▂▂


W&B run finished.

Training completed! Model saved to: Bachelors-Thesis/moderngbert_vio_models